# Introduction

This notebook fine-tunes an OpenAI Whisper model using Supervised Fine-Tuning (SFT).

Input datasets are obtained from the script `1-create-datasets.ipynb`

# ToDos

- Check all @todo in this notebook

# Set parameters, hyper-parameters

## Dataset(s)

In [ ]:
# Hugging Face datasets

# these must be available on the Hugging Face Hub
# These datasets will be used for training the model
# If more than one dataset is specified, the model will be trained on the union of all datasets
# Thes datasets must contain the following splits: 'train', 'test', 'eval'
HF_DATASETS_TRAINING = [
    "luca-r/chiara_dataset-sft_20261112" #,
#    "luca-r/chiara_dataset-sft-validated_20261014", 
]

## Model

In [ ]:
# - OpenAi Whisper starting checkpoint
# -- Local
parameter_basemodel_path = "/path/to/chiara-speech2text_models/chiara_whisper_20261026"
# -- HuggingFace Hub
#parameter_basemodel_path = "openai/whisper-tiny"
#parameter_basemodel_path = "openai/whisper-base"
#parameter_basemodel_path = "openai/whisper-small"
#parameter_basemodel_path = "openai/whisper-medium"
#parameter_basemodel_path = "openai/whisper-large"
#parameter_basemodel_path = "openai/whisper-large-v3"
#parameter_basemodel_path = "openai/whisper-large-v3-turbo"
#parameter_basemodel_path = f"{HF_USER}/chiara_whisper-large-v3-turbo_20260624"


# - Metric for model evaluation
# choose between Word Error Rate (WER) and Levenshtein Distance
parameter_trainer_metric = "wer"
#parameter_trainer_metric = "levenshtein_ratio"


# - N bins for features extractor
#  parameter_model_n_mel_bins=80 is the standard for openai/whisper-small. This is compatibile with FasterWhisper inference
#  parameter_model_n_mel_bins=128 is the standard for openai/whisper-large-v3-turbo. ?This is NOT compatible with FasterWhisper inference?
parameter_model_n_mel_bins = 128


# - ASR model language
# from transformers.models.whisper import tokenization_whisper
# languages_list = list(tokenization_whisper.TO_LANGUAGE_CODE.values()) # list of supported languages
parameter_model_language  = 'it'


# - Print info
print(f"Base model: {parameter_basemodel_path}")
print(f"Metric:     {parameter_trainer_metric}")
print(f"Language:   {parameter_model_language}")

## Trainer

In [ ]:
# - parameter_trainer_batch_size
# this directly affects footprint on the GPU RAM
# Depending on available RAM, it is possible that you will encounter a CUDA "out-of-memory" error when you start training.
# if short on GPU RAM, reduce this value incrementally by factors of 2
# (and potentially employ gradient_accumulation_steps to compensate)
#parameter_trainer_batch_size = 1    # for whisper-large-v3, on Google Colab-hosted T4 GPU
#parameter_trainer_batch_size = 16   # batch_size=16 equals to  ~90 GB of GPU RAM (for whisper-large-v3, on nvidia dgx spark)
parameter_trainer_batch_size = 18  # batch_size=18 equals to  ~99 GB of GPU RAM (for whisper-large-v3, on nvidia dgx spark)
#parameter_trainer_batch_size = 20  # batch_size=20 system crashes "out of memory"


# - parameter_trainer_n_epochs default is 3
# This parameter has a big influence on training time
# Plots show that after ~3 epochs, the models converge
parameter_trainer_n_epochs = 2
#parameter_trainer_n_epochs = 0.01 # <--- for quick tests


# - parameter_trainer_learning_rate
parameter_trainer_learning_rate = 1e-5


# - parameter_trainer_warmup_steps
parameter_trainer_warmup_steps  = 10


# - eval_batch_size
parameter_trainer_eval_batch_size = 8


# - max_gen_len
# set this as short as possible for your data
# max_length to use on each evaluation loop when predict_with_generate=True.
# Will default to the max_length value of the model configuration.
parameter_trainer_max_gen_len       = 48    # 2026.11.12: changed. was 32 before


# - layers to be updated
# For personalization, we typically only want to update the encoder and projection layer.
# Updating the decoder layer may lead to overfitting.
parameter_trainer_update_encoder    = True
parameter_trainer_update_decoder    = True
parameter_trainer_update_projection = True


# - early stopping
parameter_early_stopping_threshold  = 0.05     # minimum improvement required
#parameter_early_stopping_threshold   = 50       # <--- for quick tests

parameter_early_stopping_patience   = 20       # stop after N evaluations without improvement above 'parameter_early_stopping_threshold'
#parameter_early_stopping_patience    = 1         # <--- for quick tests

## Directories

In [ ]:
# Directory for output model
DIR_MODELS_OUTPUT           = f"/path/to/chiara-speech2text_models/temp"

# Directories for Whisper logs and output
import os
DIR_WHISPER_LOG     = os.path.join(DIR_MODELS_OUTPUT, 'whisper-logs')
DIR_WHISPER_OUTPUT  = os.path.join(DIR_MODELS_OUTPUT, 'whisper-output')

## Credentials

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/path/to/chiara-speech2text_envs/.env"

# Init notebook

In [ ]:
# Install packages

%pip install --quiet pip
%pip install --quiet datasets[audio] accelerate evaluate jiwer tensorboard
%pip install --quiet datasets
%pip install --quiet tokenizers
%pip install --quiet librosa
%pip install --quiet transformers
%pip install --quiet ipywidgets
%pip install --quiet huggingface_hub
%pip install --quiet cupy-cuda13x
%pip install --quiet Levenshtein # https://huggingface.co/spaces/Natooz/levenshtein

# run "nvidia-smi" and check CUDA version
# setup torch with the matching CUDA version, e.g. cu130 for CUDA 13.0
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# Import variables from .env file

# Load the variables from the .env file into the environment
from dotenv import load_dotenv
load_dotenv(ENV_FILE)

In [ ]:
# Load Hugging Face variables from environment
import os
HF_USER     = os.environ.get("HF_USER")
HF_TOKEN    = os.environ.get("HF_TOKEN")

#from google.colab import userdata
#HF_USER="luca-r"
#HF_TOKEN = userdata.get('HF_TOKEN')

print(f"I will use the HF user: {HF_USER}, token: {HF_TOKEN}")

In [ ]:
# Imports

import os
import soundfile as sf
import numpy as np
from datetime import datetime
from dataclasses import dataclass
import json
import numbers
from datasets import DatasetDict, Dataset

import ipywidgets as widgets
from IPython.display import Audio
from IPython.display import display

import evaluate
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor
from transformers import (
    EarlyStoppingCallback
)
from transformers.models.whisper import tokenization_whisper

import torch
from accelerate import Accelerator

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError # Import HfHubHTTPError directly from utils

In [ ]:
# get today's date in format YYYYMMDD
# this will be used to filename the ouptut from this notebook
from datetime import datetime
string_date = datetime.now().strftime("%Y%m%d")

In [ ]:
import os

# create directory for model
DIR_WHISPER_MODEL_OUTPUT = f"{DIR_MODELS_OUTPUT}/chiara_whisper_{string_date}"

# check if directory already exists, if so append a number to the directory name
if os.path.exists(DIR_WHISPER_MODEL_OUTPUT):
    i = 1
    while os.path.exists(DIR_WHISPER_MODEL_OUTPUT + f"_{i}"):
        i += 1
    DIR_WHISPER_MODEL_OUTPUT = DIR_WHISPER_MODEL_OUTPUT + f"_{i}"
os.makedirs(DIR_WHISPER_MODEL_OUTPUT, exist_ok=True)

In [ ]:
# remove and create folders for Whisper
!rm -rf {DIR_WHISPER_LOG}
!mkdir {DIR_WHISPER_LOG}

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_TRAINING_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import datasets
datasets.disable_progress_bars()

In [ ]:
# Login to Hugging Face (the processed dataset will be stored there)

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError

# login to HF
login(token=HF_TOKEN) # Pass the token explicitly for consistency and clarity

# Verify login by attempting to retrieve user information
try:
    HfApi().whoami()
    print("Successfully logged in to Hugging Face Hub.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Hugging Face login failed: {e}. Please ensure your 'hf_token' secret is valid and you have restarted the Colab runtime after setting it.")
except Exception as e:
    raise RuntimeError(f"An unexpected error occurred during Hugging Face login verification: {e}")

# cleanup
del HF_TOKEN

In [ ]:
# Empty GPU RAM

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_TRAINING_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import datasets
datasets.disable_progress_bars()

# Dataset

## Download from Hugging Face

In [ ]:
# download training dataset(s)

# datasets with ~10'000 samples (~15 GB) take ~2m to download and reconstruct

from datasets import DatasetDict, concatenate_datasets, load_dataset

if not HF_DATASETS_TRAINING: raise ValueError("HF_DATASETS_TRAINING must contain at least one dataset.")
datasets = [load_dataset(dataset_id) for dataset_id in HF_DATASETS_TRAINING]

# print info
for i, dataset in enumerate(datasets):
    print(f"Dataset #{i}: {HF_DATASETS_TRAINING[i]}")
    #print(f"  Entries: {len(dataset):,}")
    print(f"  Dataset: {dataset}")

## Concatenate different datasets 

In [ ]:
# check if all datasets have the same splits
splits = set(datasets[0].keys())
if any(set(dataset.keys()) != splits for dataset in datasets[1:]):
    raise ValueError("All datasets in HF_DATASETS_TRAINING must have the same splits.")

# Merge datasets into a single dataset
hf_dataset = DatasetDict({
    split: concatenate_datasets([dataset[split] for dataset in datasets])
    for split in datasets[0]
})
# hf_dataset.save_to_disk(DIR_DATA_HF)

# Train

## Configure training

In [ ]:
# Define Error function

# Levenshtein ratio:    higher = better.    Levenshtein ratio=(1-normalized_Levenshtein_distance). Levenshtein ratio = 1 means perfect match
# WER:                  higher = worse.     Higher WER means higher error rate
metric_greater_is_better_value = True if parameter_trainer_metric == "levenshtein_ratio" else False

# Define metric globally for testing
if parameter_trainer_metric == "wer":
    metric = evaluate.load("wer")
    print("Using WER metric for best model selection.")

elif parameter_trainer_metric == "levenshtein_ratio":
    metric = evaluate.load("Natooz/levenshtein")
    print("Using Levenshtein Ratio metric for best model selection.")

else:
    raise ValueError(f"Unknown metric choice: {parameter_trainer_metric}")

#
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = hf_processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str    = hf_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str   = hf_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    if parameter_trainer_metric == "wer":
        metric = evaluate.load("wer")
        score = 100 * metric.compute(predictions=pred_str, references=label_str)
        return {f"eval_{parameter_trainer_metric}": score}

    elif parameter_trainer_metric == "levenshtein_ratio":
        levenshtein_metric = evaluate.load("Natooz/levenshtein")
        levenshtein_result = levenshtein_metric.compute(predictions=pred_str, references=label_str)
        levenshtein_score = 100 * levenshtein_result['levenshtein_ratio']

        wer_metric = evaluate.load("wer")
        wer_score = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        return {
            "eval_levenshtein_ratio": levenshtein_score,
            "eval_wer": wer_score,
        }

    else:
        raise ValueError(f"Unknown metric choice: {parameter_trainer_metric}")


# Test error metric
results = metric.compute(
#    references  = ["foo",   "bar",    "pioggia"   ],
#    predictions = ["foo",   "baroo",  "chioggial" ],
    references  = ["foo"],
    predictions = ["foa"],
)

#
print(f"Selected metric: {parameter_trainer_metric}")
print(f"Is higher better?: {metric_greater_is_better_value}")
print(f"Result: {results}")

# cleanup
del results

In [ ]:
# Create processor

# --------------------------------------------------------------------------------
#                                  IMPORTANT
# --------------------------------------------------------------------------------
#
# make sure these parameters are the !same! as those in the 1-create-datasets.ipynb script
#
# --------------------------------------------------------------------------------


# Download Whisper checkpoint
print('Using Language: ', parameter_model_language)
parameter_model_task = "transcribe"
hf_processor = WhisperProcessor.from_pretrained(
    parameter_basemodel_path,
    language        = parameter_model_language,
    task            = parameter_model_task,
    num_mel_bins    = parameter_model_n_mel_bins)

In [ ]:
# Create model
# Download if not already present

hf_base_model = WhisperForConditionalGeneration.from_pretrained(parameter_basemodel_path)

In [ ]:
# Set model properties

hf_base_model = hf_base_model.to(torch.float32)

hf_base_model.generation_config.language           = parameter_model_language
hf_base_model.generation_config.task               = parameter_model_task
hf_base_model.generation_config.forced_decoder_ids = None
hf_base_model.config.forced_decoder_ids            = None

# Fix for AttributeError: 'GenerationConfig' object has no attribute 'lang_to_id'
if not hasattr(hf_base_model.generation_config, "lang_to_id"):
    if hasattr(hf_processor.tokenizer, "lang_code_to_id"):
        hf_base_model.generation_config.lang_to_id = hf_processor.tokenizer.lang_code_to_id
    else:
        # Fallback for Fast tokenizer if attribute is missing
        hf_base_model.generation_config.lang_to_id = {
            code: hf_processor.tokenizer.convert_tokens_to_ids(f"<|{code}|>")
            for code in tokenization_whisper.TO_LANGUAGE_CODE.values()
        }

# Fix for 'task_to_id'
if not hasattr(hf_base_model.generation_config, "task_to_id"):
    hf_base_model.generation_config.task_to_id = {
        task: hf_processor.tokenizer.convert_tokens_to_ids(f"<|{task}|>")
        for task in ["transcribe", "translate"]
    }

# to use gradient checkpointing
# hf_base_model.config.use_cache = False # Commented out as gradient checkpointing is causing issues

# Explicitly disable gradient checkpointing in the model config
hf_base_model.config.gradient_checkpointing_enabled = False

print('language set to:', hf_base_model.generation_config.language)

In [ ]:
# Define early-stopping callback
hf_trainer_early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience     = parameter_early_stopping_patience,
    early_stopping_threshold    = parameter_early_stopping_threshold,
)

In [ ]:
# Set trainer parameters

# parameter_trainer_save_steps = 0 # parameter_trainer_save_steps=0 does NOT seem to save the best model
parameter_trainer_save_steps = 5 # must be a multiple of eval_steps (which is 5)

# see https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/trainer#transformers.TrainingArguments
# https://huggingface.co/docs/transformers/v5.15.1/main_classes/trainer
hf_trainer_args = Seq2SeqTrainingArguments(
    output_dir                  = DIR_WHISPER_OUTPUT,
    #logging_dir                 = DIR_WHISPER_LOG,
    #
    per_device_train_batch_size = parameter_trainer_batch_size,
    gradient_accumulation_steps = 1,
    gradient_checkpointing      = False,
    fp16                        = False, #changed to False on 20251225
    num_train_epochs            = parameter_trainer_n_epochs,
    #
    lr_scheduler_type           = 'constant_with_warmup',
    learning_rate               = parameter_trainer_learning_rate,
    warmup_steps                = parameter_trainer_warmup_steps,
    #
    eval_strategy               = "steps",
    eval_steps                  = 5,
    per_device_eval_batch_size  = parameter_trainer_eval_batch_size,
    predict_with_generate       = True,
    generation_max_length       = parameter_trainer_max_gen_len,
    metric_for_best_model       = parameter_trainer_metric,
    greater_is_better           = metric_greater_is_better_value,
    #
    save_steps                  = parameter_trainer_save_steps,
    save_total_limit            = 2, # this affects footprint on disk
    save_strategy               = "best",
    logging_steps               = 1,
    load_best_model_at_end      = True,
    #
    remove_unused_columns       = False,
    push_to_hub                 = False,
    eval_on_start               = True,
    #
    do_train                    = True, # Trains the model
    do_eval                     = True, # Evaluates on the validation set during training
    do_predict                  = True, # Runs a final evaluation on the test set post-training
    # multi CPU workers
    dataloader_num_workers          = 2,
    dataloader_pin_memory           = True,
    dataloader_persistent_workers   = False,
    #
    report_to                   = ["tensorboard"],
)

#save_safetensors            = False, # added 20251225

#print(hf_trainer_args)

In [ ]:
# Set model parameters to be updated

hf_base_model.model.encoder.requires_grad_(parameter_trainer_update_encoder)
hf_base_model.model.decoder.requires_grad_(parameter_trainer_update_decoder)
hf_base_model.proj_out.requires_grad_(parameter_trainer_update_projection)

# This function counts the number of trainable parameters in the model.
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

print('encoder params to update/total:', count_trainable_parameters(hf_base_model.model.encoder), hf_base_model.model.encoder.num_parameters())
print('decoder params to update/total:', count_trainable_parameters(hf_base_model.model.decoder), hf_base_model.model.decoder.num_parameters())
print('overall # trainable parameters:', count_trainable_parameters(hf_base_model))
print('    overall # model parameters:', hf_base_model.model.num_parameters())

## Utils for SFT training

In [ ]:
# Define trainer

# ~45s required
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = ([{"input_features": feature["input_features"]} for feature in features])
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = ([{"input_ids": feature["labels"]} for feature in features])

        #with warnings.catch_warnings():
        #    warnings.simplefilter("ignore", category=UserWarning) # suppress warning about attention_mask in tokenizer.pad

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

hf_data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor               = hf_processor,
    decoder_start_token_id  = hf_base_model.config.decoder_start_token_id,
)

hf_trainer = Seq2SeqTrainer(
    args            = hf_trainer_args,
    model           = hf_base_model,
    train_dataset   = hf_dataset["train"],
    eval_dataset    = hf_dataset["test"],
    data_collator   = hf_data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [hf_trainer_early_stopping_callback],
)

# cleanup
del Any, Dict, List, Union

## Fit model (fine-tune)

In [ ]:
# Set `PYTORCH_CUDA_ALLOC_CONF`
# Setting `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` can help mitigate
# CUDA out-of-memory errors by enabling a more flexible memory allocation
# strategy in PyTorch. This allows PyTorch to reserve and manage GPU memory
# more efficiently, potentially preventing fragmentation issues that lead
# to OOM errors, especially when dealing with varying memory demands during training.

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"PYTORCH_CUDA_ALLOC_CONF set to: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")

In [ ]:
# Start tensorboard
# Tensorboard allows to show the results from the next cell. To use it:
# - Play this cell, it will show an empty output
# - Open http://localhost:6006 in your browser
# NOTE: if ssh'd, you have to ensure that port 6006 is forwarded (e.g. go to the "Ports" tab in Visual Code, and you have to manually forward port 6006)

%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir {DIR_WHISPER_OUTPUT}

In [ ]:
# The model and reference model are moved to the selected device per iteration.
device = torch.device("cuda:0")

In [ ]:
# Empty GPU RAM

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

In [ ]:
# Fine-tune model

# ...it takes time to train a model from scratch! e.g.:
# - ~2h30m for 30mins of input audio-data, 10 epochs
# - ~5h30m for 1h45mins of input audio-data, 10 epochs

# You can monitor resources with either of the following terminal commands
# watch -n 1 nvidia-smi
# btop

# @todo: fix warnings:
# - [transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
# - [transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


# define accelerator
hf_accelerator = Accelerator()

# start timer
start_time = datetime.now()

# train
hf_trainer.train()                                                                              # option 1: train  from parameter_basemodel_path
#hf_trainer.train(resume_from_checkpoint=True)                                                  # option 2: resume from latest available checkpoint
#hf_trainer.train(resume_from_checkpoint=os.path.join(DIR_WHISPER_OUTPUT, "checkpoint-855"))    # option 3: resume from specific checkpoint

# compute time
end_time = datetime.now()
time_delta = end_time - start_time

# compute training duration
metadata_training_duration_seconds  = time_delta.total_seconds()
metadata_training_duration_hours    = metadata_training_duration_seconds / 3600

# cleanup
try:
    del start_time, end_time, time_delta
    del metadata_training_duration_seconds
except: pass

In [ ]:
# Empty GPU RAM after training

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

# Evaluate

In [ ]:
# Evaluate trained model on available datasets

In [ ]:
# train split
# CAREFUL:  these results will not be representative of model's performance on unseen / new data
#           (because both these datasets have been seen during training!)

'''
print('Evaluating best model on "train" set, language:', parameter_model_language)
results_train = hf_trainer.evaluate(hf_dataset["train"], language=parameter_model_language)
print(results_train)
'''

In [ ]:
# test split
# CAREFUL:  these results will not be representative of model's performance on unseen / new data
#           (because both these datasets have been seen during training!)

'''
print('Evaluating best model on "test" set, language:', parameter_model_language)
results_test = hf_trainer.evaluate(hf_dataset["test"], language=parameter_model_language)
print(results_test)
'''

In [ ]:
# eval dataset
# This dataset has NOT been seen during training
# the results are representative of model's performance on unseen / new data

print('Evaluating best model on "eval" set, language:', parameter_model_language)
results_eval = hf_trainer.evaluate(hf_dataset["eval"], language=parameter_model_language)
print(results_eval)

# Save (model, metadata, results)

## Save model

In [ ]:
# Save the best model locally
# The trainer automatically saves the best model, so we just need to copy it

import shutil

# best model directory
!rm -rf {DIR_WHISPER_MODEL_OUTPUT}
!mkdir  {DIR_WHISPER_MODEL_OUTPUT}

# save model and processor, so we can later load as pretrained
hf_trainer.model.save_pretrained(DIR_WHISPER_MODEL_OUTPUT, safe_serialization=False)    # model
hf_processor.save_pretrained    (DIR_WHISPER_MODEL_OUTPUT, safe_serialization=False)    # processor

print(f"✓ Best model saved to: {DIR_WHISPER_MODEL_OUTPUT}")

# List saved files
saved_files = os.listdir(DIR_WHISPER_MODEL_OUTPUT)
print(f"✓ Files saved: {saved_files}")

# cleanup
del saved_files

## Save results

In [ ]:
# Metadata directory
DIR_METADATA                = os.path.join(DIR_WHISPER_MODEL_OUTPUT, "metadata")
DIR_METADATA_MODEL          = os.path.join(DIR_METADATA, "model")
DIR_METADATA_DATASETS       = os.path.join(DIR_METADATA, "datasets")
DIR_METADATA_TENSORBOARD    = os.path.join(DIR_METADATA, "tensorboard")
!rm -rf {DIR_METADATA}
!mkdir  {DIR_METADATA}
!mkdir  {DIR_METADATA_MODEL}
!mkdir  {DIR_METADATA_DATASETS}
!mkdir  {DIR_METADATA_TENSORBOARD}

In [ ]:
# Save evaluation results

import json

# Save evaluation results

results_test_file_path = os.path.join(DIR_METADATA_MODEL, 'results_trainer.json')

# Combine results into a single dictionary structure
evaluation_results = {
    #"train":    results_train,
    #"test":     results_test,
    "eval":     results_eval
}

# Write the evaluation results to the file in JSON format
with open(results_test_file_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, indent=4, ensure_ascii=False)

print(f"Evaluation results saved to: {results_test_file_path}")

# cleanup
del f, evaluation_results
del results_test_file_path
#del results_test, results_train, results_eval, results_test_file_path

## Save training parameters

In [ ]:
# Save training arguments file

hf_trainer_args_dict = hf_trainer_args.to_dict()

# Define the path for the training arguments file
hf_trainer_args_file_path = os.path.join(DIR_METADATA_MODEL, 'trainer_parameters.json')

# Save the dictionary to a JSON file for human-readable format
with open(hf_trainer_args_file_path, 'w') as f:
    json.dump(hf_trainer_args_dict, f, indent=4)

print(f"Training arguments saved to: {hf_trainer_args_file_path}")

# cleanup
del f

## Save model metadata

In [ ]:
# simplify basemodel name
base_model_name = parameter_basemodel_path.rsplit('/', 1)[-1]
print(base_model_name)

In [ ]:
# Save Metadata file for model

# Get current time in HHhMMm format
metadata_training_date  = datetime.now().strftime('%Y%m%d')
metadata_finish_time    = datetime.now().strftime('%Hh%Mm')

# get info on GPU
if torch.cuda.is_available():
    metadata_gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Type: {metadata_gpu_name}")
else:
    metadata_gpu_name = "n/a (no GPU found, or CUDA not available)"

# Build field history: prepend previous values if the base checkpoint
# already has a metadata_model.json (keeps a chain of all ancestors)
metadata_base_model         = base_model_name
metadata_training_technique = "Supervised Fine-Tuning (SFT)"

prev_metadata_path = os.path.join(parameter_basemodel_path, "metadata", "model", "metadata_model.json")
if not os.path.isfile(prev_metadata_path):
    prev_metadata_path = os.path.join(parameter_basemodel_path, "metadata", "metadata_model.json")
prev_metadata = None
if os.path.isfile(prev_metadata_path):
    with open(prev_metadata_path, "r") as f:
        prev_metadata = json.load(f)
    
    '''prev_base_model = prev_metadata.get("base_model(s)")
    if prev_base_model:
        metadata_base_model = f"{prev_base_model} > {base_model_name}"
        print(f"Base model history: {metadata_base_model}")
    
    prev_training_technique = prev_metadata.get("training_technique(s)")
    if prev_training_technique:
        metadata_training_technique = f"{prev_training_technique} > {metadata_training_technique}"
        print(f"Training technique history: {metadata_training_technique}")'''

# Create metadata for this model and retain the complete base-model metadata
current_metadata = {
    "base_model":                   metadata_base_model,
    "training_technique(s)":        metadata_training_technique,
    "training_date":                metadata_training_date,
    "training_finish_time":         metadata_finish_time,
    "training_duration_hours":      f"{metadata_training_duration_hours:.2f}",
    "model_language":               parameter_model_language,
    #"model_type":                  parameter_basemodel_path,
    "model_n_mel_bins":             parameter_model_n_mel_bins,
    "model_task":                   parameter_model_task,
    "model_metric":                 hf_trainer_args.metric_for_best_model,
    "trainer_batch_size":           parameter_trainer_batch_size,
    "trainer_learning_rate":        parameter_trainer_learning_rate,
    "trainer_warmup_steps":         parameter_trainer_warmup_steps,
    "trainer_max_gen_len":          parameter_trainer_max_gen_len,
    "trainer_n_epochs":             parameter_trainer_n_epochs,
    "trainer_update_decoder":       parameter_trainer_update_decoder,
    "trainer_update_encoder":       parameter_trainer_update_encoder,
    "trainer_update_projection":    parameter_trainer_update_projection,
    "gpu_name":                     metadata_gpu_name
}
metadata = {
    **current_metadata,
    "metadata_history": {
        "base_model": prev_metadata,
    }
}

# Define the path for the JSON file
metadata_file_path = os.path.join(DIR_METADATA_MODEL, 'metadata_model.json')

# Save the dictionary to a JSON file
with open(metadata_file_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Model metadata saved to: {metadata_file_path}")

# cleanup
del f, metadata_file_path, metadata, current_metadata, metadata_base_model, metadata_training_technique, prev_metadata_path, prev_metadata

In [ ]:
# get latest "trainer_state.json" from checkpoint folders
# and remove all temp checkpoints

# Find the latest checkpoint directory
latest_checkpoint_dir   = None
target_checkpoint_path  = None
max_checkpoint_num      = -1

# List contents of OUTPUT_DIR
output_dir_contents = os.listdir(DIR_WHISPER_OUTPUT)
print(DIR_WHISPER_OUTPUT)
print(output_dir_contents)

for item in output_dir_contents:
    if item.startswith('checkpoint-') and os.path.isdir(os.path.join(DIR_WHISPER_OUTPUT, item)):
        print(f'Found checkpoint folder: {item}')
        try:
            checkpoint_num = int(item.split('-')[1])
            if checkpoint_num > max_checkpoint_num:
                max_checkpoint_num = checkpoint_num
                latest_checkpoint_dir = item
        except ValueError:
            # Not a valid checkpoint number, ignore
            pass

if latest_checkpoint_dir:
    source_checkpoint_path = os.path.join(DIR_WHISPER_OUTPUT, latest_checkpoint_dir)
    print(f"Found latest checkpoint folder: {source_checkpoint_path}")

    # Define the target path, naming the folder 'checkpoint'
    target_checkpoint_path = os.path.join(DIR_WHISPER_OUTPUT, 'checkpoint_latest')

    # Ensure the target directory in Google Drive exists and is empty for a clean copy
    if os.path.exists(target_checkpoint_path):
        shutil.rmtree(target_checkpoint_path)
    os.makedirs(target_checkpoint_path, exist_ok=True)

    # Copy only .json files from source to target
    if os.path.exists(source_checkpoint_path):
        copied_files_count = 0
        json_files_copied = []
        for file_name in os.listdir(source_checkpoint_path):
            source_file_path = os.path.join(source_checkpoint_path, file_name)
            if os.path.isfile(source_file_path) and file_name.endswith('.json'):
                target_file_path = os.path.join(target_checkpoint_path, file_name)
                shutil.copy2(source_file_path, target_file_path)
                copied_files_count += 1
                json_files_copied.append(file_name)

        if copied_files_count > 0:
            print(f"Copied {copied_files_count} JSON files ({', '.join(json_files_copied)}) from checkpoint folder to: {target_checkpoint_path}")
        else:
            print(f"No JSON files found to copy from {source_checkpoint_path}.")
    else:
        print(f"Error: Checkpoint folder not found at {source_checkpoint_path}")
else:
    print(f"No checkpoint directory found in {DIR_WHISPER_OUTPUT}. Skipping checkpoint folder save.")

# remove all files except "trainer_state.json" from target_checkpoint_path
try:
    if os.path.exists(target_checkpoint_path):
        for file_name in os.listdir(target_checkpoint_path):
            if file_name != "trainer_state.json":
                os.remove(os.path.join(target_checkpoint_path, file_name))

    # move "trainer_state.json"
    source_file_path = os.path.join(target_checkpoint_path, "trainer_state.json")
    target_file_path = os.path.join(DIR_METADATA_MODEL, "trainer_state.json")
    shutil.move(source_file_path, target_file_path)
    print(f'Moved "trainer_state.json" to: {target_file_path}')
except Exception as e:
    print(f"Error while cleaning up checkpoint folder: {e}")

In [ ]:
# tensorboard files

# copy content of DIR_WHISPER_OUTPUT/runs to DIR_METADATA/tensorboard
shutil.copytree(os.path.join(DIR_WHISPER_OUTPUT, "runs"), DIR_METADATA_TENSORBOARD, dirs_exist_ok=True)
print(f"Tensorboard logs copied to: {DIR_METADATA_TENSORBOARD}")

## Save dataset(s) metadata

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

for dataset_id in HF_DATASETS_TRAINING:
    dataset_name = dataset_id.rsplit("/", 1)[-1]

    # metadata.json
    metadata_path = hf_hub_download(
        repo_id     = dataset_id,
        filename    = "metadata.json",
        repo_type   = "dataset",
    )
    shutil.copy2(metadata_path, os.path.join(DIR_METADATA_DATASETS, f"ds_{dataset_name}_metadata.json"))
    print(f"Copied metadata for {dataset_id}")

    # data.json
    data_path = hf_hub_download(
        repo_id     = dataset_id,
        filename    = "data.json",
        repo_type   = "dataset",
    )
    shutil.copy2(data_path, os.path.join(DIR_METADATA_DATASETS, f"ds_{dataset_name}_data.json"))
    print(f"Copied data for {dataset_id}")   

## Evaluate

Use the notebook `models-evaluator/models-evaluator.ipynb` to evaluate model more in detail

The notebook also prepare a results file that can be further analyzed with the `nvidia nemo data explorer` tool.

See: https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/speech_data_explorer.html

## Push to HF hub

This script does not automatically push to the HF hub.

This is to avoid automatically uploading models which performance have not been manually assessed / confirmed by a human developer.

The developer is supposed to assess the performance of the trained model against old models (e.g. through the script `models-evaluator/models-evaluator.ipynb`)

If performace are satisfactory, one can then manually upload the model through script `utils/push-model-to-hf-hub.py`

# Cleanup

In [ ]:
# exit()

In [ ]:
'''
del checkpoint_num, copied_files_count, file_name
del latest_checkpoint_dir, max_checkpoint_num
del item, json_files_copied, output_dir_contents
del source_checkpoint_path, target_checkpoint_path
del target_file_path, source_file_path
'''

In [ ]:
'''
del metadata_training_date, metadata_finish_time
del metadata_gpu_name
del metadata_training_duration_hours
'''

In [ ]:
'''
del parameter_trainer_batch_size, parameter_trainer_n_epochs
del parameter_trainer_learning_rate, parameter_trainer_warmup_steps
del parameter_trainer_max_gen_len, parameter_trainer_save_steps
del parameter_trainer_update_decoder, parameter_trainer_update_encoder, parameter_trainer_update_projection
'''

In [ ]:
'''
del parameter_model_task, parameter_trainer_metric
del parameter_model_language, parameter_basemodel_path, parameter_model_n_mel_bins
'''

In [ ]:
'''
del metric, metric_greater_is_better_value
del hf_accelerator, hf_base_model, hf_data_collator
del hf_dataset, hf_processor, hf_trainer, hf_trainer_args, hf_trainer_args_dict, hf_trainer_args_file_path
del hf_trainer_early_stopping_callback
'''

In [ ]:
'''
del parameter_early_stopping_patience, parameter_early_stopping_threshold
'''

In [ ]:
'''
import shutil

# remove log and output folders
try:
    shutil.rmtree(DIR_WHISPER_LOG)
    shutil.rmtree(DIR_WHISPER_OUTPUT)
    print(f'Removed log folder: {DIR_WHISPER_LOG}')
    print(f'Removed output folder: {DIR_WHISPER_OUTPUT}')
except: pass
'''

In [ ]:
'''
#del DIR_DATA_HF, DIR_DATA_HF_TEST, DIR_DATA_HF_TRAIN
del DIR_WHISPER_LOG, DIR_WHISPER_MODEL_OUTPUT
'''

# Next steps

Once model is trained, it can be used to transcribe audios to text.

e.g.:
- The model can be deployed on a server that accepts '.wav' audio files and returns a text transcript, see `docs/1-setup.md`
- The model can be loaded from another notebook to evaluate its performance, see `models-evaluator/models-evaluator.ipynb`